# 03 — Trevisan's Taxonomy and the Consistency Principle

Synthesise the opening arc of Module 4: turn the two examples you built into a *law* every quantum optimal transport must obey, then place the competing definitions of QOT on one map.

**Prerequisites:** `04/01_diagonal_collapse`, `04/02_noncommuting_same_diagonal`.

**What you'll be able to do**
- State the **diagonal-collapse consistency principle**: a legitimate quantum OT must reduce to classical OT whenever the two states commute and the cost operator is diagonal in their shared eigenbasis.
- Confirm the principle numerically on a commuting (diagonal) pair, where classical OT gives the answer every candidate must match.
- Read **Trevisan's taxonomy** — the landscape of non-equivalent QOT formulations — and see why this course follows the coupling path.
- **Open** the quantum-optimal-transport row of the classical ↔ quantum dictionary.

You arrive with two examples in hand. In `04/01` you watched classical OT call $|+\rangle\langle+|$ and $I/2$ identical because they share a $Z$-diagonal; in `04/02` you sharpened that into a *non-commuting* pair, $|+_x\rangle\langle+_x|$ and $|+_y\rangle\langle+_y|$, and read the commutator's Frobenius norm as a dial for non-commutation — one that, as `04/02` warned, does not measure how much a given diagonal discards. This synthesis does two things with that work. First it extracts the *positive* lesson hiding inside the negative examples — a requirement that pins down what any honest quantum OT must do on the easy (commuting) cases. Then it steps back to the whole field and lays the competing definitions side by side, so you can see where the object this module builds sits among them.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import ot

from qot_course import viz
from qot_course.colors import COLORS
from qot_course.infotheory.quantum import bures_distance
from qot_course.quantum_ot.preview import (
    commutativity_norm,
    diagonal_in_computational_basis,
)
from qot_course.transport.discrete import cityblock_cost

np.random.seed(42)
viz.use_course_style()

## 1. The consistency principle — classical OT is the commuting boundary

Both examples so far were *negative*: they showed classical OT failing on quantum states. Turn them over and a *positive* requirement appears. The trouble in `04/01`–`04/02` was always the same — coherence the $Z$-diagonal cannot see. In `04/02` a non-zero commutator came with it; in `04/01` it did not, because that pair commutes and the $Z$ basis is not its common eigenbasis. So ask the opposite question: *when does that trouble vanish entirely?*

It vanishes when the two states **commute** and you read their diagonals in a basis they share. If $[\rho, \sigma] = 0$ then a single basis diagonalises both at once, and in that shared eigenbasis each one genuinely *is* a probability vector — there are no off-diagonal coherences left to discard. On commuting states the diagonal reduction, taken in that shared eigenbasis, loses nothing about the two states. The commutator norm you built in `04/02` is the gauge of this: it reads exactly zero precisely when such a shared basis exists.

That observation is strong enough to become a design rule, provided we say what it asks of the **cost** as well as of the states:

> **Diagonal-collapse consistency principle.** Take two density matrices that commute, and a cost operator that is diagonal in the product of their shared eigenbasis, $C = \sum_{ij} c_{ij}\,|i\rangle\langle i| \otimes |j\rangle\langle j|$. Then quantum optimal transport must return *classical* optimal transport on the common eigenvalues, with the cost table $c_{ij}$. Classical OT is the **commuting boundary** of quantum OT.

The condition on the cost is part of the principle. In the coupling formulation this course follows, the argument takes three lines: erase the off-diagonal entries of a coupling in the product eigenbasis; its marginals do not change, because they are diagonal; its cost does not change, because $C$ is diagonal; and what remains is a classical coupling. When the cost operator has off-diagonal entries in the shared eigenbasis, the argument breaks at the third step, and a coupling that carries coherence can cost strictly *less* than every classical plan, even between two states that commute. We will meet both situations in the next two notebooks.

Trevisan's survey makes the same point from the other side: recovering the classical Wasserstein distance on diagonal states is a property it calls useful when diagonal states encode classical probabilities, and one that it notes does not hold in general across formulations. So this principle is the baseline that a formulation *together with its cost* must pass before it can be called an extension of a given classical problem. We confirm it on a concrete commuting pair: the number classical OT returns is the number such a quantum OT is then obliged to match.

In [ ]:
# Two density matrices, both diagonal in the Z basis -> they commute (a classical pair).
p_vec = np.array([0.6, 0.4])
q_vec = np.array([0.1, 0.9])
rho_p = np.diag(p_vec).astype(complex)
rho_q = np.diag(q_vec).astype(complex)

print("rho_p, rho_q are both diagonal in the Z basis:")
print(f"  diag(rho_p) = {diagonal_in_computational_basis(rho_p).tolist()}")
print(f"  diag(rho_q) = {diagonal_in_computational_basis(rho_q).tolist()}")
print(f"  commutator norm ||[rho_p, rho_q]||_F = {commutativity_norm(rho_p, rho_q):.2e}"
      "   (zero -> they commute)")
print()

# On this commuting pair, classical OT is the CORRECT answer -- the consistency target.
# Ground cost on the two outcomes {0, 1}: C[i, j] = |i - j|.
cost = cityblock_cost([0, 1], [0, 1])
classical_w1 = ot.emd2(p_vec, q_vec, cost)
print(f"classical W_1 on the diagonals = {classical_w1:.4f}")
print()
print("The consistency principle: a quantum OT whose cost operator is diagonal in this")
print("shared basis MUST return this same number for this commuting pair. The quadratic")
print("position cost of 04/05 passes the test; the SWAP cost of 04/04 is not diagonal here.")

**Read the output.** The two states are diagonal in the same basis, so their commutator norm is zero to machine precision — by the gauge of `04/02`, this is a *classical* pair with no hidden coherence. On exactly this kind of pair classical OT is trustworthy: feeding the diagonals $(0.6, 0.4)$ and $(0.1, 0.9)$ to the Kantorovich solver returns $0.5000$, the cost of moving the mass that differs between them. The consistency principle now reads as a constraint with a *number* attached: a quantum OT whose cost operator is diagonal in this basis, with the same cost table, must return this same $0.5000$ for $\rho_p$ and $\rho_q$.

Here is how the two costs of this module fare, so that nothing surprises you later. The quadratic position cost of `04/05` is diagonal in the $Z$ product basis, and the coupling SDP returns $0.5000$ with it: the test is passed. The SWAP cost $I - \mathrm{SWAP}$ of `04/04` has the same diagonal (cost $0$ on $|00\rangle$ and $|11\rangle$, cost $1$ on $|01\rangle$ and $|10\rangle$) but it also has off-diagonal entries, and its optimal coupling uses them: it carries a coherence of $0.2449$ between $|01\rangle$ and $|10\rangle$ and brings the value down to $0.2101$. Both are legitimate transport problems. Only the first one extends this classical problem.

### See the boundary — what changes when you step off it

The principle speaks about one point: commuting states and a diagonal cost. To see what happens next to that point, keep $\rho_p = \mathrm{diag}(0.6, 0.4)$ and give $\rho_q$ a real coherence $c$ while keeping its diagonal at $(0.1, 0.9)$:

$$\rho_q(c) = \begin{pmatrix} 0.1 & c \\ c & 0.9 \end{pmatrix}, \qquad 0 \le c \le 0.3 .$$

At $c = 0$ you are back on the commuting pair above; at $c = 0.3$ the state is pure. For each $c$ we compute three numbers: the commutator norm, the classical $W_1$ on the $Z$-diagonals, and the Bures distance of `02/12`, a state-level ruler that sees the whole operator.

In [ ]:
# Step off the commuting boundary: give rho_q a real coherence c, keep its diagonal fixed.
coherences = np.linspace(0.0, 0.3, 61)  # c = 0.3 is the largest value that keeps rho_q positive
comm_norms, w1_diag, bures = [], [], []
for c in coherences:
    rho_qc = np.array([[0.1, c], [c, 0.9]], dtype=complex)
    comm_norms.append(commutativity_norm(rho_p, rho_qc))
    w1_diag.append(ot.emd2(p_vec, diagonal_in_computational_basis(rho_qc).real, cost))
    bures.append(bures_distance(rho_p, rho_qc))

# On the boundary, Bures is the Hellinger-type distance of 03/13, not the transport value.
hellinger_form = np.sqrt(2 * (1 - np.sum(np.sqrt(p_vec * q_vec))))
print(f"c = 0   : W_1 on diagonals = {w1_diag[0]:.4f}   Bures = {bures[0]:.4f}   "
      f"Hellinger form sqrt(2(1 - sum sqrt(p q))) = {hellinger_form:.4f}")
print(f"c = 0.3 : W_1 on diagonals = {w1_diag[-1]:.4f}   Bures = {bures[-1]:.4f}   "
      f"commutator norm = {comm_norms[-1]:.4f}")

fig, (ax_left, ax_right) = plt.subplots(1, 2, figsize=(11, 4.5))
ax_left.plot(coherences, comm_norms, color=COLORS["highlight"], linewidth=2.0)
ax_left.set_xlabel(r"coherence $c$ of $\rho_q$")
ax_left.set_ylabel(r"$\|[\rho_p, \rho_q(c)]\|_F$")
ax_left.set_title("Commutator norm: zero only at c = 0", pad=12)

ax_right.plot(coherences, w1_diag, color=COLORS["source"], linewidth=2.0,
              label=r"classical $W_1$ on the $Z$-diagonals")
ax_right.plot(coherences, bures, color=COLORS["quantum"], linewidth=2.0,
              label="Bures distance")
ax_right.set_xlabel(r"coherence $c$ of $\rho_q$")
ax_right.set_ylabel("distance")
ax_right.set_title("The diagonal stays put; the state does not", pad=12)
ax_right.legend()
plt.tight_layout()
plt.show()

**Read the figure.** On the left, the commutator norm is zero at $c = 0$ and grows in a straight line as soon as $\rho_q$ carries coherence: only the left edge of the plot is on the commuting boundary. On the right, the classical $W_1$ on the $Z$-diagonals stays flat at $0.5000$ for every $c$, because the diagonal of $\rho_q$ never moves. The Bures distance does move: it climbs from $0.5569$ to $0.8390$ as $\rho_q$ goes from the commuting pair to a pure state. Off the boundary the diagonal answer has stopped describing the states, and the principle makes no claim there.

Look also at the left edge, where the principle does apply. There the two curves start at different heights, $0.5000$ against $0.5569$. Bures is a legitimate distance between the states, and on commuting states it becomes the Hellinger-type distance $\sqrt{2(1 - \sum_i \sqrt{p_i q_i})}$ of `03/13`, which the cell checks against the closed form. It is not a transport cost, so it is not the number the consistency principle asks for. The principle concerns transport problems, and the transport value it asks for here is $0.5000$.

## 2. Trevisan's taxonomy — the landscape of quantum OT

The consistency principle narrows the field but does not single out one winner: *many* constructions reduce to classical OT on commuting states while disagreeing with one another off it. Quantum optimal transport is therefore not a single object but a small family of non-equivalent definitions, each lifting a different facet of the classical theory. Trevisan's survey is the map of this landscape. The table below keeps four of its families and adds the entropic one, which the survey does not treat, with the notebook in this module where each idea appears.

| Formulation | Core idea | Where in the course |
|---|---|---|
| **Quantum couplings (SDP)** | Replace a classical coupling $P$ by a bipartite density matrix $\rho_{AB}$ whose partial traces are the two marginals; minimise $\mathrm{tr}(C\,\rho_{AB})$ (Golse, Mouhot & Paul, 2016; as a finite-dimensional SDP, Cole, Eckstein, Friedland & Życzkowski, 2023). | **04/04 onward** (the path this module takes) |
| **Entropic / quantum Sinkhorn** | Add a von-Neumann-entropy regulariser and iterate matrix exponentials with partial traces, exactly as classical Sinkhorn iterates a Gibbs kernel. | later in Module 4 |
| **Dynamic / Carlen–Maas** | Replace the continuity equation by a Lindblad-type generator and define a Wasserstein metric on the *dynamics* of density matrices. | later in Module 4 (brief) |
| **Channel-based** | Minimise over the quantum channels that send one state to the other (De Palma & Trevisan, 2021). The matching couplings carry a *transposed* marginal, so this is a different object from the coupling SDP. | survey only |
| **Qubit-$W_1$** | Specialise the Lipschitz (Kantorovich dual) side to multi-qubit systems with a Hamming-type metric (De Palma, Marvian, Trevisan, Lloyd, 2021). | survey only |

**Different formulations, different theorems.** There is no single "*the*" quantum optimal transport, and reading these as rivals is the wrong picture — they answer different questions, and they do not all recover classical OT on commuting states: the survey notes that this property fails in general, the coupling distance with a position-and-momentum cost being its example. This course commits to the **coupling SDP** as its principal object. The reason is continuity with everything you have already built: it is the most direct analogue of the Kantorovich linear program from Module 3, it inherits a clean entropic regularisation (the quantum Sinkhorn iteration), and the Amari projection picture of Module 3 has a quantum counterpart (`04/08`). The Bures distance of Module 2 stays beside it as a separate ruler: on commuting states it is a Hellinger-type distance, not the transport value, as the figure above showed. The other entries stay on the map as the wider context that Trevisan's survey lays out.

## 3. The dictionary opens its quantum-OT row

One thread has run through the whole course: the classical ↔ quantum **dictionary**, built row by row, each row in the notebook that earns it. Module 2 filled the *information-geometry* rows — entropy, relative entropy, mutual information, the Fisher/Bures metric. Module 3 opened the *transport* rows — the coupling, the Kantorovich LP, the transportation polytope, the Birkhoff/assignment special case — and wrote their quantum side in words only: none of those quantum objects had been built yet.

This arc is what lets us start building that side. The consistency principle fixes the *boundary condition* (the quantum entries must reduce to the classical ones when states commute), and Trevisan's taxonomy names the *object* that goes in each cell. So the quantum-OT row of the dictionary **opens here**. The transport rows of Module 3 — the coupling, the Kantorovich LP, the transportation polytope, the Wasserstein distance — carry their quantum partners in the complete table as commitments the rest of the module will honour and verify, starting with the coupling in `04/04`.

**New in the dictionary.** The complete table lives in [the course dictionary](../dictionary.ipynb) (`notebooks/dictionary.ipynb`).

| Classical | Quantum |
| --- | --- |
| commuting / diagonal states | the consistency boundary: quantum OT $=$ classical OT when the cost operator is diagonal in their shared eigenbasis |
| $W_1 = \int \lvert F_\mu - F_\nu\rvert\,\mathrm{d}x$ | qubit-$W_1$: Lipschitz dual, no integral form (not built in this course) |

The first row is the one this notebook establishes, on the commuting pair above. The second sets the $W_1$ you read as the area between two cumulative curves in `03/06` beside the qubit-$W_1$ of the taxonomy above. That quantum distance starts from the Lipschitz dual side; the course does not compute it, as the row says, and you will meet it again in a theorem in `04/19`. With the boundary condition pinned and the object named, the quantum side of the transport rows now has a law to obey and a construction to follow; it is *open*, and the next notebook begins to build it.

## Your turn

**Warm-up.** Build your own commuting pair and read off its consistency target. Pick two probability vectors on three outcomes, say $a = (0.5, 0.3, 0.2)$ and $b = (0.2, 0.5, 0.3)$, form the diagonal density matrices $\rho_a = \mathrm{diag}(a)$ and $\rho_b = \mathrm{diag}(b)$, and confirm with `commutativity_norm` that they commute. Then state — *before solving* — the number a quantum OT with a cost diagonal in this basis must return for this pair, and check it with `ot.emd2` against a `cityblock_cost` on the outcomes $\{0, 1, 2\}$.

**Go further.** The consistency principle is a statement about the *commuting* boundary only. Take the non-commuting equator pair from `04/02` ($|+_x\rangle\langle+_x|$ and $|+_y\rangle\langle+_y|$) and argue, in two or three sentences, why the principle says *nothing* about the number quantum OT should return for it. Which quantity from `04/02` certifies that this pair lies strictly off the boundary?

**Challenge.** Place one formulation on the map yourself. Read section 4 of Trevisan's survey (DOI:10.1007/s40574-024-00428-5) and pick any single formulation from the taxonomy table other than the coupling SDP. In a short paragraph, state what classical object it lifts, name the one structural feature that distinguishes it from the coupling SDP, and explain how it still respects the consistency principle in its own commuting limit.

## What you built

- You turned the two negative examples of `04/01`–`04/02` into a positive law — the **diagonal-collapse consistency principle**: on commuting states, and with a cost operator that is diagonal in their shared eigenbasis, a quantum OT must reduce to classical OT, because that is exactly where the diagonal discards nothing.
- You confirmed the principle on a commuting diagonal pair, where the commutator norm is zero and classical OT returns the consistency target $0.5000$ — the number a quantum OT with that diagonal cost is obliged to match.
- You read **Trevisan's taxonomy**, the map of non-equivalent QOT formulations, and saw why this course follows the **coupling SDP**: it is the closest analogue of the Kantorovich LP and carries the entropic and information-geometric structure forward.
- You **opened the quantum-OT row** of the classical ↔ quantum dictionary: the consistency boundary is its first secured entry, the qubit-$W_1$ stands beside the classical $W_1$, and the coupling, the SDP, the polytope and the distance wait in the complete table as commitments for the rest of the module.

This is the keystone of the module's opening arc. Two notebooks ago the gap between classical and quantum transport was a puzzle on a single qubit; now it is a precise requirement with a number attached, set inside a map of the whole field. You know what a quantum OT must do, you know the family of objects that try to do it, and you know which one this course will build.

## What's next

In `04/04_quantum_coupling_sdp` the coupling formulation stops being a row in a table and becomes a concrete computation: we define the quantum coupling, set up the semidefinite program, and solve it first with the SWAP cost, which sees the coherence classical OT could not. The boundary condition you pinned down here is met one notebook later, in `04/05`, when the quadratic position cost enters.

## References

- D. Trevisan, "Quantum optimal transport: an invitation", *Bollettino dell'Unione Matematica Italiana* **18**, 347–360 (2025). DOI:10.1007/s40574-024-00428-5
- F. Golse, C. Mouhot & T. Paul, "On the mean field and classical limits of quantum mechanics", *Communications in Mathematical Physics* **343**, 165–205 (2016). DOI:10.1007/s00220-015-2485-7 — the first transport problem posed over quantum couplings.
- S. Cole, M. Eckstein, S. Friedland & K. Życzkowski, "On quantum optimal transport", *Mathematical Physics, Analysis and Geometry* **26**, 14 (2023). DOI:10.1007/s11040-023-09456-7 — arXiv:2105.06922. The transport problem over all quantum couplings, solved as a semidefinite program, with the SWAP-based cost $\tfrac12(I - \mathrm{SWAP})$; this course uses $I - \mathrm{SWAP}$, twice that cost.
- G. De Palma & D. Trevisan, "Quantum optimal transport with quantum channels", *Annales Henri Poincaré* 22, 3199–3234 (2021). DOI:10.1007/s00023-021-01042-3
- G. De Palma, M. Marvian, D. Trevisan & S. Lloyd, "The quantum Wasserstein distance of order 1", *IEEE Transactions on Information Theory* 67, 6627–6643 (2021). DOI:10.1109/TIT.2021.3076442
- E. A. Carlen & J. Maas, "An analog of the 2-Wasserstein metric in non-commutative probability under which the Fermionic Fokker–Planck equation is gradient flow for the entropy", *Communications in Mathematical Physics* 331, 887–926 (2014). DOI:10.1007/s00220-014-2124-8

**Previous:** `notebooks/04_QuantumOptimalTransport/02_noncommuting_same_diagonal.ipynb` · **Next:** `notebooks/04_QuantumOptimalTransport/04_quantum_coupling_sdp.ipynb`